# Importing Libraries and Setup

In [1]:
import os

os.environ["HF_HOME"] = f"/rs1/researchers/a/amallav/models/hf_home"
os.environ["HF_HUB_CACHE"] = os.path.join(os.environ["HF_HOME"], "hub")
os.environ["HF_HUB_OFFLINE"] = "1"   # only after cache is populated

In [2]:
import json #Used later to store the top-k predictions as a JSON string in the final CSV
from pathlib import Path
import pandas as pd
import torch
from bioclip import TreeOfLifeClassifier, Rank #TreeOfLifeClassifier = the classifier that predicts biological taxa; Rank = tells the classifier what taxonomic level you want

/usr/local/usrapps/ftrscape/snair3/env_AiPipeline/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/rs1/researchers/a/amallav/")).resolve()
OUTPUTS_DIR=Path(os.environ.get("OUTPUTS_DIR", "/rs1/researchers/a/amallav/results")).resolve()

OUTPUTS_SAM_DIR = Path(os.environ.get("OUTPUTS_SAM_DIR", OUTPUTS_DIR / "outputs_sam")).resolve()

OUTPUTS_BCSAM_DIR = Path(os.environ.get("OUTPUTS_BCSAM_DIR", OUTPUTS_DIR / "outputs_bioclip_sam")).resolve()
OUTPUTS_BCSAM_DIR.mkdir(parents=True, exist_ok=True)

# CROP_OUT = PROJECT_DIR / "outputs_crop"
META_CSV = OUTPUTS_SAM_DIR / "metadata.csv" #This is the metadata file that tells us what crop files exist.
OUT_CSV = OUTPUTS_BCSAM_DIR / "bioclip_species_predictions.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR, "| exists:", OUTPUTS_DIR.exists())
print("OUTPUTS_SAM_DIR:", OUTPUTS_SAM_DIR, "| exists:", OUTPUTS_SAM_DIR.exists())
print("OUTPUTS_BCSAM_DIR:", OUTPUTS_BCSAM_DIR, "| exists:", OUTPUTS_BCSAM_DIR.exists())
print("META_CSV    :", META_CSV, "| exists:", META_CSV.exists())
print("OUT_CSV     :", OUT_CSV, "| exists:", OUT_CSV.exists())

PROJECT_DIR: /rs1/researchers/a/amallav
OUTPUTS_DIR: /rs1/researchers/a/amallav/results | exists: True
OUTPUTS_SAM_DIR: /rs1/researchers/a/amallav/results/outputs_sam | exists: True
OUTPUTS_BCSAM_DIR: /rs1/researchers/a/amallav/results/outputs_bioclip_sam | exists: True
META_CSV    : /rs1/researchers/a/amallav/results/outputs_sam/metadata.csv | exists: True
OUT_CSV     : /rs1/researchers/a/amallav/results/outputs_bioclip_sam/bioclip_species_predictions.csv | exists: False


# Load Bioclip 

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
# model_dir = "/share/ftrscape/{}/models/bioclip".format(__import__("os").environ["USER"])

In [6]:
MODEL_STR = "hf-hub:imageomics/bioclip-2"

TOP_K = 5 #top 5 species predictions for each image
BATCH_SIZE = 1 #This controls how many cropped images are processed at once. Might have to tune this parameter

classifier = TreeOfLifeClassifier(
    device=device,
    model_str=MODEL_STR,
)

print("Device   :", device)
print("Model    :", MODEL_STR)
print("Top-k    :", TOP_K)
print("Batch size:", BATCH_SIZE)

Device   : cuda
Model    : hf-hub:imageomics/bioclip-2
Top-k    : 5
Batch size: 1


# Load CSV

In [7]:
assert META_CSV.exists(), f"metadata.csv not found: {META_CSV}"

meta = pd.read_csv(META_CSV)

required_cols = {"image", "masked_crop_path"}
missing_cols = required_cols - set(meta.columns)
assert not missing_cols, f"metadata.csv is missing columns: {missing_cols}"

In [10]:
meta

,image,sam_score,x1,y1,x2,y2,mask_path,masked_crop_path,crop_exists
0,/share/ftrscape/image_dataset/obs_102691691_ph...,0.942442,8.658074,29.890091,1358.724976,514.842712,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
1,/share/ftrscape/image_dataset/obs_102691691_ph...,0.863153,318.028046,565.133301,1104.258301,1622.248779,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
2,/share/ftrscape/image_dataset/obs_102691691_ph...,0.914058,15.059799,353.937561,1283.214111,1684.873291,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
3,/share/ftrscape/image_dataset/obs_10317797_pho...,0.978479,485.393738,514.087158,1146.264038,1254.859253,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
4,/share/ftrscape/image_dataset/obs_130480768_ph...,0.968434,973.297546,583.725098,1511.949707,812.199951,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
...,...,...,...,...,...,...,...,...,...
238,/share/ftrscape/image_dataset/obs_341067936_ph...,0.947607,826.501343,464.825287,1210.934326,815.939087,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
239,/share/ftrscape/image_dataset/obs_341194952_ph...,0.967287,626.544006,1032.230713,1329.089233,1904.813232,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
240,/share/ftrscape/image_dataset/obs_341194952_ph...,0.961827,629.947754,1043.745850,1312.430786,1715.669434,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True
241,/share/ftrscape/image_dataset/obs_341194952_ph...,0.984406,1332.358765,803.885620,1627.071289,1800.349976,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,True


In [8]:
# #converts paths into absolute paths.
# def resolve_path(p):
#     if pd.isna(p):
#         return None
#     p = str(p).strip()
#     if os.path.isabs(p):
#         return p
#     return str((PROJECT_DIR / p).resolve())

# meta["masked_crop_path"] = meta["masked_crop_path"].apply(resolve_path)

In [9]:
meta = meta[meta["masked_crop_path"].notna()].copy() #remove rows where crop path is missing
meta["crop_exists"] = meta["masked_crop_path"].apply(os.path.exists)

missing_count = (~meta["crop_exists"]).sum()
if missing_count > 0:
    print(f"Skipping {missing_count} rows because crop file does not exist.")

meta = meta[meta["crop_exists"]].copy()  #only rows where the crop file actually exists.



In [11]:
keep_cols = ["image", "masked_crop_path"]
if "rank" in meta.columns:
    keep_cols.append("rank")

crop_df = meta[keep_cols].drop_duplicates(subset=["masked_crop_path"]).reset_index(drop=True) #creates a clean crop table without deduplicates

print("Total valid cropped images:", len(crop_df))
crop_df.head()

Total valid cropped images: 243


,image,masked_crop_path
0,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...
1,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...
2,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...
3,/share/ftrscape/image_dataset/obs_10317797_pho...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...
4,/share/ftrscape/image_dataset/obs_130480768_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...


# Run species classification on all cropped images

In [12]:
crop_paths = crop_df["masked_crop_path"].tolist() #Collect all crop image paths into a list.

predictions = classifier.predict(
    images=crop_paths, #Pass the cropped image files to the model.
    rank=Rank.SPECIES, #Predict at species level.
    k=TOP_K, #Return the top 5 predictions for each image.
    batch_size=BATCH_SIZE, #Process 16 images at a time.
)

pred_df = pd.DataFrame(predictions) #Turns the prediction output into a table.

print("Total prediction rows:", len(pred_df))
pred_df.head()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 243/243 [00:27<00:00,  8.70images/s]


Total prediction rows: 1215


,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score
0,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Chromista,Ochrophyta,Phaeophyceae,Laminariales,Lessoniaceae,Egregia,menziesii,Egregia menziesii,,0.901757
1,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Chromista,Ochrophyta,Phaeophyceae,Scytosiphonales,Scytosiphonaceae,Analipus,japonicus,Analipus japonicus,,0.009070
2,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Chromista,Ochrophyta,Phaeophyceae,Laminariales,Laminariaceae,Hedophyllum,sessile,Hedophyllum sessile,,0.007255
3,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Animalia,Mollusca,Gastropoda,,Lottiidae,Discurria,insessa,Discurria insessa,,0.004901
4,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Chromista,Ochrophyta,Phaeophyceae,Laminariales,Alariaceae,Alaria,marginata,Alaria marginata,,0.004806


# Convert top-k predictions into one row per cropped image

In [13]:
assert "file_name" in pred_df.columns, "Expected 'file_name' in prediction output"
assert "score" in pred_df.columns, "Expected 'score' in prediction output"
assert "species" in pred_df.columns, "Expected 'species' in prediction output"

In [20]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    for i in range(5):
        no=i+1
        no=str(no)
        val="top"+no+"_species"
        sc="top"+no+"_score"
        cc="top"+no+"_common_name"
        row = {
            "masked_crop_path": crop_path,
            # val:group.iloc[i].get("species"),
            "species": group.iloc[i].get("species"),
            # "top2_species": top2.get("species"),
            # "top3_species": top3.get("species"),
            # "top4_species": top4.get("species"),
            # "top5_species": top5.get("species"),
            "common_name": group.iloc[i].get("common_name"),
            "score": group.iloc[i].get("score"),
            "top_no":no,
            "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

final_df = crop_df.merge(summary_df, on="masked_crop_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(final_df))
final_df.head()

Final rows: 1215


,image,masked_crop_path,species,common_name,score,top_no,topk_predictions_json
0,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Egregia menziesii,,0.901757,1,"[{""species"": ""Egregia menziesii"", ""common_name..."
1,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Analipus japonicus,,0.009070,2,"[{""species"": ""Egregia menziesii"", ""common_name..."
2,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Hedophyllum sessile,,0.007255,3,"[{""species"": ""Egregia menziesii"", ""common_name..."
3,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Discurria insessa,,0.004901,4,"[{""species"": ""Egregia menziesii"", ""common_name..."
4,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Alaria marginata,,0.004806,5,"[{""species"": ""Egregia menziesii"", ""common_name..."


In [21]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    row = {
        "masked_crop_path": crop_path,
        "top1_species": top1.get("species"),
        "top2_species": top2.get("species"),
        "top3_species": top3.get("species"),
        "top4_species": top4.get("species"),
        "top5_species": top5.get("species"),
        "top1_common_name": top1.get("common_name"),
        "top1_score": top1.get("score"),
        "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

full_df = crop_df.merge(summary_df, on="masked_crop_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Full top rows:", len(full_df))
full_df.head()

Full top rows: 1215


,image,masked_crop_path,top1_species,top2_species,top3_species,top4_species,top5_species,top1_common_name,top1_score,topk_predictions_json
0,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Egregia menziesii,Analipus japonicus,Hedophyllum sessile,Discurria insessa,Alaria marginata,,0.901757,"[{""species"": ""Egregia menziesii"", ""common_name..."
1,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Egregia menziesii,Marginariella urvilliana,Marginariella boryana,Alaria marginata,Alaria praelonga,,0.846646,"[{""species"": ""Egregia menziesii"", ""common_name..."
2,/share/ftrscape/image_dataset/obs_102691691_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Egregia menziesii,Alaria marginata,Alaria nana,Alaria praelonga,Alaria crispa,,0.797723,"[{""species"": ""Egregia menziesii"", ""common_name..."
3,/share/ftrscape/image_dataset/obs_10317797_pho...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Larus glaucescens,Larus occidentalis,Larus californicus,Larus smithsonianus,Evasterias troschelii,Glaucous-winged gull,0.860455,"[{""species"": ""Larus glaucescens"", ""common_name..."
4,/share/ftrscape/image_dataset/obs_130480768_ph...,/gpfs_common/share03/ftrscape/snair3/wew_noteb...,Dina lineata,Anisorhynchodemus luteicollis,Erpobdella nigricollis,Asiaticobdella fenestrata,Myzobdella lugubris,,0.146216,"[{""species"": ""Dina lineata"", ""common_name"": """"..."


# Save final output CSV

In [23]:
OUT_CSV = OUTPUTS_DIR / "results.csv"
final_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /gpfs_common/share03/ftrscape/snair3/wew_notebooks/outputs_bioclip/results.csv


In [24]:
OUT_CSV = OUTPUTS_DIR / "bioclip_species_predictions.csv"
full_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /gpfs_common/share03/ftrscape/snair3/wew_notebooks/outputs_bioclip/bioclip_species_predictions.csv
